In [ ]:
import pandas as pd
import numpy as np

# 加载 Meta Ads 导出的 CSV
ads_path = "Firmoo-ES-Ads-Nov-12-2025-Nov-18-2025.csv"

ads = pd.read_csv(ads_path, encoding='utf-8')
ads.head()

In [ ]:
# === 1. 如果有 Ad ID 就重命名，否则创建空列 ===
if 'Ad ID' in ads.columns:
    ads = ads.rename(columns={'Ad ID': 'ad_id'})
else:
    ads['ad_id'] = np.nan

# === 2. 统一数字列格式 ===
numeric_cols = [
    'Reach',
    'Impressions',
    'Frequency',
    'CPM (cost per 1,000 impressions) (USD)',
    'Cost per results',
    'Purchases',
    'Purchase ROAS (return on ad spend)',
    'Amount spent (USD)',
    'Results'
]

for col in numeric_cols:
    if col in ads.columns:
        ads[col] = (
            ads[col]
            .astype(str)
            .str.replace(',', '', regex=False)
            .str.replace('--', '', regex=False)
        )
        ads[col] = pd.to_numeric(ads[col], errors='coerce')

# === 3. 简化列名
ads = ads.rename(columns={
    'Purchase ROAS (return on ad spend)': 'roas',
    'Amount spent (USD)': 'spend_usd',
    'Cost per results': 'cpa',
    'CPM (cost per 1,000 impressions) (USD)': 'cpm',
    'Purchases': 'purchases'
})


In [ ]:
# === 1. 只看 active 广告 ===
active_ads = ads[ads['Ad delivery'] == 'active'].copy()

# === 2. 基于你西班牙站点的优化业务阈值 ===
good_roas = 2.5
ok_roas   = 1.5
min_pur_good = 30
min_pur_ok   = 10
low_spend    = 150
freq_fatigue = 2.5   # 频次超过这个认为素材疲劳上升

# === 3. 用当前批次的中位数做“动态基准线” ===
med_roas = active_ads['roas'].median()
med_cpa  = active_ads['cpa'].median()
med_cpm  = active_ads['cpm'].median()

In [ ]:
# === 正向评分：ROAS 等越大越好 ===
def norm_ratio_pos(x, baseline, max_ratio=3.0):
    if baseline is None or baseline <= 0 or pd.isna(baseline):
        return 0.0
    if pd.isna(x) or x <= 0:
        return 0.0
    r = x / baseline
    r = max(0.0, min(r, max_ratio))
    return r / max_ratio

# === 反向评分：CPA、CPM 等越小越好 ===
def norm_ratio_inverse(x, baseline, max_ratio=3.0):
    if baseline is None or baseline <= 0 or pd.isna(baseline):
        return 0.0
    if pd.isna(x) or x <= 0:
        return 0.0
    r = baseline / x
    r = max(0.0, min(r, max_ratio))
    return r / max_ratio

# === Meta Ranking 文案映射 ===
quality_map = {
    'Above average': 1,
    'Average': 0,
    'Below average - Bottom 35% of ads': -1,
    'Below average - Bottom 20% of ads': -1,
    '-': 0
}

# === 质量评分（0~1）===
def calc_quality_score(row):
    vals = []
    for col in ['Quality ranking', 'Engagement rate ranking', 'Conversion rate ranking']:
        v = row.get(col, '-')
        vals.append(quality_map.get(v, 0))
    avg = sum(vals) / len(vals)
    return (avg + 1) / 2  # 映射到 0~1

In [ ]:
def score_one_ad(row):
    if row['Ad delivery'] != 'active':
        return pd.Series({'ad_score': 0.0, 'decision': '非投放中'})
    
    roas = row['roas']
    cpa  = row['cpa']
    spend = row['spend_usd']
    pur   = row['purchases']
    freq  = row['Frequency']
    cpm   = row['cpm']
    
    # 各维度得分
    roas_s = norm_ratio_pos(roas, med_roas)
    cpa_s  = norm_ratio_inverse(cpa, med_cpa)
    cpm_s  = norm_ratio_inverse(cpm, med_cpm)
    volume_s = 0.0 if pd.isna(pur) else min(pur / min_pur_good, 1.0)
    spend_s  = 0.0 if pd.isna(spend) else min(spend / low_spend, 1.0)
    quality_s = calc_quality_score(row)
    
    # 频次疲劳惩罚
    if pd.isna(freq):
        freq_penalty = 0.0
    else:
        over = max(0.0, freq - freq_fatigue)
        freq_penalty = min(over / 3.0, 1.0)
    
    # 综合评分
    base_score = (
        0.4 * roas_s +
        0.2 * cpa_s +
        0.1 * cpm_s +
        0.1 * volume_s +
        0.1 * spend_s +
        0.1 * quality_s
    )
    
    ad_score = base_score * (1 - 0.5 * freq_penalty)
    ad_score = max(0, min(ad_score, 1))
    
    # 决策文案
    if (spend < low_spend) and (pd.isna(pur) or pur < min_pur_ok):
        decision = '数据不足，继续小额测试'
    else:
        if ad_score >= 0.7 and roas >= good_roas and pur >= min_pur_good:
            decision = '强烈建议加预算'
        elif ad_score >= 0.45 and roas >= ok_roas and pur >= min_pur_ok:
            decision = '建议继续投放并优化'
        elif ad_score >= 0.3:
            decision = '可继续小额测试 / 观察'
        else:
            decision = '建议暂停或重做素材'
    
    return pd.Series({'ad_score': ad_score, 'decision': decision})

In [ ]:
# === 对 active 广告整体打分 ===
scored_ads = active_ads.copy()

# 运行评分函数
scored_ads[['ad_score', 'decision']] = scored_ads.apply(score_one_ad, axis=1)

# === 定义决策优先级（从高到低） ===
decision_priority = {
    '强烈建议加预算': 1,
    '建议继续投放并优化': 2,
    '可继续小额测试 / 观察': 3,
    '数据不足，继续小额测试': 4,
    '建议暂停或重做素材': 5,
    '非投放中': 6
}

# 映射到每条广告
scored_ads['decision_priority'] = scored_ads['decision'].map(decision_priority)

# === 按业务优先级排序，内部再按 ad_score 排序 ===
scored_ads = (
    scored_ads
    .sort_values(['decision_priority', 'ad_score'], ascending=[True, False])
    .reset_index(drop=True)
)

# === 设置展示字段 ===
cols_to_show = [
    'ad_id',
    'Ad name',
    'Ad set name',
    'Ad delivery',
    'Reach',
    'Impressions',
    'Frequency',
    'spend_usd',
    'purchases',
    'roas',
    'cpa',
    'cpm',
    'ad_score',
    'decision'
]

result_ads_view = scored_ads[cols_to_show]

# === 展示前 30 条广告（业务优先级排序后） ===
result_ads_view.head(30)
